# CMDF training and evaluation — leakage-free, pre-registered

This notebook replaces `train_test_evaluate_cmdf_final.ipynb`. What changed and why:

| Original notebook | This version | Why |
|---|---|---|
| one 5-fold split | 10 × repeated stratified 5-fold (cohort × label), folds identical to the habitat study | single splits of 62 patients gave AUROC SDs of 0.12–0.29 |
| best of 6 models picked on the test folds and reported | one pre-specified model family (L2 logistic regression, inner-CV tuning, training-only threshold); primary comparison fixed in advance | picking the best of many on the same folds inflates the estimate |
| accuracy / F1 on an imbalanced positive class | AUROC with bootstrap CIs, AUPRC, calibration, sensitivity/specificity at a training-only threshold | accuracy and F1 reward predicting the majority class |
| no significance testing | Nadeau–Bengio corrected t-test, pairing null, label permutation, BH correction | the question is whether CMDF adds anything, not whether a number is high |
| VAE / 1D-CNN / attention-MIL | not used (n = 74 in the primary set) | deep models are not identifiable at this sample size |
| final model refit on all data and exported | not exported | a deployable model needs external validation first |
| — | methodological arm: the original protocol re-run on the same features, and on permuted labels | shows how much of the original performance was optimism |

In [ ]:
# Run inside WSL with the habitat study's environment (~/venvs/habitat2; add ipykernel to it to use this notebook).
# Paths default to the workspace layout; override with GLIOMA_DATA_ROOT, HABITAT_OUTPUT_DIR and CMDF_OUTPUT_DIR.
import sys
from pathlib import Path
CMDF_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CMDF_DIR))

import pandas as pd
from IPython.display import Image, Markdown, display
from cmdf1p19q.config import build_cmdf_config
from cmdf1p19q.study import CMDFStudy

hcfg, ccfg = build_cmdf_config()
study = CMDFStudy(hcfg, ccfg)
study.protocol_sha = __import__("habitat1p19q.reporting", fromlist=["sha256"]).sha256(CMDF_DIR / "PROTOCOL.md")
print("data:", hcfg["data_root"], "| habitat cache:", hcfg["output_dir"], "| outputs:", ccfg["output_dir"])

## 1–3. Cohort, cached features and feature blocks (see notebook 01; cached runs take about 2 minutes)

In [ ]:
study.stage_data()
study.stage_voxel()
study.stage_blocks()

## 4. Cross-validation (UTSW and pooled)

In [ ]:
study.stage_cv()

## 5. Primary (P+D vs P, UTSW) and secondary comparisons, H2

In [ ]:
study.stage_analyses()

## 6. Null tests: pairing null and label permutation

In [ ]:
study.stage_nulls()

## 7. Cross-cohort transport (UTSW → UCSF)

In [ ]:
study.stage_loco()

## 8. Robustness, ablation, calibration

In [ ]:
study.stage_robustness()

## 9. Methodological arm: the original notebooks' protocol

In [ ]:
study.stage_naive()

## 10. Figures, conclusion and manifest

In [ ]:
study.stage_report()
for f in sorted((ccfg["output_dir"] / "figures").glob("*.png")):
    display(Image(filename=str(f)))
display(Markdown((ccfg["output_dir"] / "conclusion.md").read_text(encoding="utf-8")))